# Распределённые системы агентов для высоконагруженных сред

> Внимание! Материал ноутбука подходит для работы в Google Colaboratory. Мы не можем гарантировать стабильную работу кода на личных устройствах и на других системах виртуализации.

## Установка зависимостей

Установим необходимые библиотеки

In [1]:
# %pip install -q "deepagents>=0.5,<0.6" "gigachat>=0.2,<0.3" "langchain>=1.3,<2" "langchain-gigachat>=0.5,<0.6" "langgraph>=1.2,<2" "python-dotenv>=1,<2"

#  Добавьте GIGACHAT_CREDENTIALS в панель Colab «Секреты» и разрешите
#  notebook доступ к нему. Затем раскомментируйте следующие три строки.

import os
# from google.colab import userdata
# os.environ["GIGACHAT_CREDENTIALS"] = userdata.get("GIGACHAT_CREDENTIALS")
from dotenv import load_dotenv
load_dotenv()

from pprint import pprint

## Введение

### Постановка проблемы

В третьем занятии локальный `AgentHarness` обрабатывал план в одном процессе, а внешний эффект оставлял предложением. После подтверждения реальный сервис часто передаёт такую операцию отдельному рабочему процессу. Между запросом и обработчиком появляется брокер сообщений: он хранит задачу, выдаёт её рабочему процессу и может выдать снова после сбоя.

Модель и контур агента решают, **что** предложить; очередь и обработчик гарантируют, **как** безопасно выполнить уже разрешённую задачу. Поэтому обычного `try/except` недостаточно.

Семинар отвечает на три практических вопроса: кто владеет задачей во время обработки, когда сообщение можно окончательно удалить из очереди и как безопасно выдать ту же задачу снова после сбоя.

### Цели занятия

После выполнения работы вы сможете объяснить жизненный цикл задачи от публикации до `ack`, реализовать ограниченные повторы с DLQ, остановить приём работы при занятом поставщике и безопасно обработать повторную выдачу задачи после сбоя.

### Предварительные знания

Нужны dataclass, словари, исключения и обычные функции. Термины «аренда» (`lease`), DLQ и идемпотентность вводятся ниже с нуля. Шардирование и контрольные точки LangGraph в обязательную часть не входят.

### Итоговый артефакт

Небольшая среда выполнения: приоритетная очередь выдаёт задачу в аренду, ограничитель поставщика сдерживает нагрузку, обработчик сохраняет идемпотентный результат, а цикл обработки делает `ack` только после успеха.

### Участники и жизненный цикл задачи

Worker обрабатывает **задачу (`Task`)**. `Delivery` — лишь служебная запись о том, какая это попытка выдачи задачи.

| Понятие | Что означает | Для чего нужно |
|---|---|---|
| Отправитель (`publisher`) | Компонент, который создаёт задачу и помещает её в очередь. | Отделить приём запроса от его фонового выполнения. |
| Брокер и очередь | Система, которая хранит готовые задачи и выдаёт их обработчикам. | Сгладить нагрузку и не потерять работу при временной занятости обработчиков. |
| Рабочий процесс (`worker`) | Процесс, который получает задачу и выполняет обработчик. | Выполнять задачи независимо и параллельно в заданных пределах. |
| Поставщик (`provider`) | Внешняя служба или модель, к которой обращается рабочий процесс. | Получить результат, недоступный внутри очереди. |
| Задача (`Task`) | Предметные данные и служебные параметры работы. | Хранить, что именно требуется выполнить. |
| Попытка доставки (`Delivery`) | Служебная запись с задачей и номером попытки; факт текущей аренды хранит брокер в `inflight`. | Отдельно учитывать каждую выдачу одной задачи. |
| Приоритетная очередь | Очередь, которая сначала выдаёт более важные задачи. | Обслуживать срочную работу раньше обычной. |
| Аренда (`lease`) | Закрепление задачи за рабочим процессом в `inflight`. В учебном коде нет часов и TTL: исчезновение обработчика имитирует ручной `recover()`. | Не выдать одну попытку двум рабочим процессам одновременно и вернуть её после сбоя. |
| Подтверждение (`ack`) | Сообщение брокеру об успешной обработке задачи. | Закрыть текущую Delivery и удалить работу из очереди. |
| Отказ (`nack`) | Отрицательное завершение попытки. Отдельного метода `nack` здесь нет: его роль после временной ошибки играет `retry_or_dead_letter()`. | Запланировать повторную доставку или перевод в очередь ошибок. |
| Доставка как минимум один раз | Гарантия, при которой задача не теряется, но может быть выдана повторно. | Переживать сбои между выполнением операции и подтверждением. |

## 0. Теория: пять участников одного запроса

Продолжим сквозной случай `P-77`: после проверки правил и подтверждения появилась команда «вернуть 1490 ₽ по платежу P-77». В локальной программе её могла бы выполнить одна функция. В распределённой системе ответственность разделена:

| Участник | Что хранит или решает | Чего не делает |
|---|---|---|
| **Отправитель** | создаёт `Task` и публикует её | не выбирает свободный рабочий процесс |
| **Брокер сообщений** | хранит очередь, приоритет и аренду задачи | не определяет успешность бизнес-операции |
| **Рабочий процесс** | временно получает задачу и вызывает обработчик | не удаляет сообщение до сохранения результата |
| **Ограничитель обращений к поставщику** | ограничивает одновременные внешние вызовы | не отвечает за повторы сообщения |
| **Хранилище результатов** | помнит результат и выполненную бизнес-команду | не управляет очередью |

Главная мысль: рабочий процесс **не владеет** задачей навсегда. Он берёт её в аренду у брокера. Пока нет `ack`, брокер обязан считать задачу незавершённой.

## Задача и попытка её доставки

`Task` — сама работа, например «вернуть 1490 ₽». Именно её обрабатывает рабочий процесс. `Delivery` — не отдельная работа, а служебная запись о конкретной попытке выдать эту задачу: в ней лежат `Task` и номер `attempt`.

Дальше слово **«задача»** означает предметную работу, а **«доставка»** используется только для технической попытки её передачи.

Короткий словарь:

- **аренда (`lease`)** — временное право рабочего процесса обрабатывать задачу;
- **`ack`** — подтверждение брокеру, что задача успешно обработана, а текущую Delivery можно закрыть;
- **повторная попытка** — новый запуск после ожидаемой временной ошибки;
- **DLQ (`Dead Letter Queue`)** — очередь необработанных сообщений, исчерпавших допустимое число попыток;
- **повторная доставка** — повторная выдача той же задачи после сбоя.

Учебная аренда намеренно упрощена: часов и TTL здесь нет. `reserve()` переносит попытку в `inflight`, а ручной `recover()` имитирует обнаружение исчезнувшего рабочего процесса. После ожидаемой временной ошибки метод `retry_or_dead_letter()` играет роль `nack`: планирует повтор или переносит задачу в очередь ошибок.

`ack` — не синоним фразы «обработчик начал работу». Он означает: «результат уже сохранён, повтор больше не нужен».

## Почему повторы нормальны: доставка как минимум один раз

Учебный брокер даёт гарантию **доставки как минимум один раз** (`at-least-once`): опубликованная задача не должна молча исчезнуть, но иногда может прийти больше одного раза. Классическое окно сбоя:

1. рабочий процесс выполнил возврат;
2. сохранил результат;
3. остановился за мгновение до `ack`;
4. брокер не увидел `ack` и выдал сообщение снова.

Брокер не может угадать, была ли выполнена бизнес-операция. Поэтому выполнение операции ровно один раз обеспечивает обработчик через идемпотентность, а не очередь.

| Идентификатор | На какой вопрос отвечает |
|---|---|
| `message_id` | «Это повтор того же сообщения?» |
| `idempotency_key` | «Это та же бизнес-команда, возможно в новом сообщении?» |

Например, `m-101` и `m-102` могут быть двумя сообщениями, но оба иметь `idempotency_key="refund:P-77"`. Сохранённые результаты у них разные, а операция возврата должна быть выполнена только один раз.

## Три границы безопасности

Вся практика ниже проверяет три границы:

| Граница | Ошибка без неё | Правило |
|---|---|---|
| Брокер | сообщение потеряно после раннего удаления | `reserve` до работы, `ack` после сохранения |
| Поставщик | рабочие процессы создают неограниченную нагрузку | при занятом месте остановиться или ограничить очередь ожидания |
| Бизнес-операция | повторная доставка повторяет списание или возврат | операцию защищает `idempotency_key` |

В рамках занятия есть четыре коротких задания. Первые три реализуют правила по отдельности, четвёртое связывает их в один проход рабочего процесса. Продвинутые темы перечислены в конце и не нужны для зачёта.

## Галерея проверяемых сценариев

| Сценарий | Ожидаемый результат | Граница |
|---|---|---|
| срочная и обычная задачи | срочная выдаётся первой | приоритетная очередь |
| временная ошибка поставщика | попытки 1→2→3, затем DLQ | ограниченные повторы задачи |
| место занято, очередь ожидания свободна/полна | `buffered` / `dropped_overload` | ограничение нагрузки |
| сбой после сохранения результата и до ack | повтор без повторной операции | устранение дублей и идемпотентность |
| новый message_id той же бизнес-команды | новый результат, прежняя операция | отдельный idempotency_key |

Последнее задание объединяет только уже знакомые функции. Шардирование, квоты и оркестрация с контрольными точками оставлены для следующего уровня, чтобы не смешивать их с базовым жизненным циклом задачи.

### Понятия безопасной обработки очереди

| Понятие | Что означает | Для чего нужно |
|---|---|---|
| Повторная доставка (`redelivery`) | Новая выдача той же задачи после ошибки или `recover()`; в промышленном брокере её также запускает истечение аренды. | Продолжить работу после сбоя обработчика. |
| Очередь ошибок (`DLQ`) | Отдельное хранилище задач, исчерпавших допустимые попытки. | Не блокировать основную очередь неисправимой задачей и сохранить её для разбора. |
| Временная ошибка | Сбой, после которого повтор может завершиться успешно. | Вернуть задачу в очередь в пределах бюджета попыток. |
| Неисправимая задача (`poison message`) | Задача, которая стабильно падает из-за своих данных или логики. | Остановить бесконечные повторы и отправить задачу в DLQ. |
| Ограничение нагрузки (`backpressure`) | Отказ или ожидание при заполненном буфере или исчерпанной параллельности. | Не передать поставщику больше работы, чем он способен обработать. |
| Предел параллельности | Максимальное число одновременно выполняемых запросов. | Защитить ресурсы приложения и поставщика. |
| Буфер | Ограниченное число задач, ожидающих свободного места. | Пережить короткий всплеск без неограниченного роста памяти. |
| Идемпотентность | Свойство, при котором повтор одной операции не меняет результат после первого успеха. | Безопасно обрабатывать повторные доставки. |
| Ключ идемпотентности | Стабильный идентификатор одной предметной операции. | Узнать повтор и вернуть уже сохранённый результат. |
| Дедупликация | Поиск и подавление повторного выполнения по ключу. | Не создать второй платёж, обращение или сообщение. |
| Восстановление после сбоя | Возврат просроченных доставок и чтение сохранённых результатов. | Продолжить обработку после перезапуска рабочего процесса. |

## Сквозной пример перед кодом

Пусть отправитель публикует задачу:

```python
Task(
    message_id="m-101",
    priority=1,
    idempotency_key="refund:P-77",
    payload="вернуть 1490 ₽ по P-77",
)
```

Штатный путь состоит из пяти шагов:

1. брокер принимает `Task`, создаёт первую `Delivery` и помещает её в `ready`;
2. рабочий процесс делает `reserve`, а запись о попытке переходит в `inflight`;
3. ограничитель обращений к поставщику разрешает внешний вызов;
4. обработчик сохраняет бизнес-операцию по `refund:P-77` и результат по `m-101`;
5. цикл обработки делает `ack`.

Если поставщик временно недоступен, выполняется ограниченная повторная попытка. Если рабочий процесс останавливается после шага 4, брокер повторно выдаёт ту же задачу: обработчик находит сохранённые ключи, не повторяет возврат и позволяет циклу обработки завершить `ack`.

Дальше код собирает именно этот сценарий, по одному правилу за задание.

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field, replace
from heapq import heappop, heappush
from typing import Any, Callable


class TransientError(RuntimeError):
    """Временная ошибка: задачу можно выдать повторно в пределах лимита попыток."""


class WorkerCrash(RuntimeError):
    """Рабочий процесс остановился после сохранения результата, но до ack."""


@dataclass(frozen=True, slots=True)
class Task:
    """Описывает одну предметную задачу, которую отправитель публикует в очередь."""

    # message_id различает сообщения, idempotency_key — одну бизнес-команду.
    message_id: str
    priority: int
    idempotency_key: str
    payload: str


@dataclass(frozen=True, slots=True)
class Delivery:
    """Хранит задачу и номер конкретной попытки её выдачи обработчику."""

    task: Task
    attempt: int = 1

pprint({
    "поля_задачи": list(Task.__dataclass_fields__),
    "поля_попытки_Delivery": list(Delivery.__dataclass_fields__),
    "классы_ошибок": [TransientError.__name__, WorkerCrash.__name__],
})

{'классы_ошибок': ['TransientError', 'WorkerCrash'],
 'поля_задачи': ['message_id', 'priority', 'idempotency_key', 'payload'],
 'поля_попытки_Delivery': ['task', 'attempt']}


Task хранит саму задачу, а Delivery — ссылку на неё и номер конкретной попытки выдачи; состояние аренды остаётся у брокера.

## 1. Очередь и один цикл обработки

Отправитель кладёт `Task` в ограниченную приоритетную очередь. Брокер сам создаёт первую `Delivery`, а `reserve()` возвращает её рабочему процессу. Меньшее значение `priority` означает более срочную задачу. До `ack` брокер держит текущую попытку в `inflight`.

При `TransientError` для задачи создаётся следующая попытка. После исчерпания лимита запись о задаче переходит в DLQ. Неожиданный `WorkerCrash` не считается обработанной попыткой.

In [3]:
class PriorityTaskQueue:
    """Учебный брокер: приоритетная очередь, аренда, повторы и DLQ."""

    def __init__(self, capacity: int):
        """Создаёт очередь с общей границей для готовых и арендованных задач."""
        if type(capacity) is not int or capacity <= 0:
            raise ValueError("capacity должен быть положительным целым")
        self.capacity = capacity
        self.ready: list[tuple[int, int, Delivery]] = []
        self.inflight: dict[str, Delivery] = {}
        self.dead_letters: list[Delivery] = []
        self._sequence = 0

    def _push(self, delivery: Delivery) -> None:
        """Возвращает задачу в ready, сохраняя номер попытки и порядок FIFO."""
        self._sequence += 1
        # Меньшее число означает больший приоритет; sequence сохраняет FIFO.
        heappush(self.ready, (delivery.task.priority, self._sequence, delivery))

    def publish(self, task: Task) -> bool:
        """Принимает новую задачу или возвращает False при заполненной очереди."""
        # Арендованная задача тоже занимает capacity: она может вернуться.
        if len(self.ready) + len(self.inflight) >= self.capacity:
            return False
        # Первая Delivery появляется внутри брокера, а не у отправителя.
        self._push(Delivery(task))
        return True

    def reserve(self, worker_id: str) -> Delivery | None:
        """Выдаёт следующую задачу и возвращает запись о попытке Delivery."""
        if worker_id in self.inflight:
            raise ValueError("рабочий процесс уже держит задачу")
        if not self.ready:
            return None
        delivery = heappop(self.ready)[2]
        # До ack брокер хранит задачу в аренде у конкретного рабочего процесса.
        self.inflight[worker_id] = delivery
        return delivery

    def ack(self, worker_id: str) -> None:
        """Подтверждает успешно сохранённый результат и закрывает аренду."""
        if worker_id not in self.inflight:
            raise ValueError("нет арендованной задачи для ack")
        self.inflight.pop(worker_id)

    def retry_or_dead_letter(self, worker_id: str, max_attempts: int) -> str:
        """Планирует новую попытку задачи или переносит её запись в DLQ."""
        delivery = self.inflight.pop(worker_id)
        if delivery.attempt >= max_attempts:
            self.dead_letters.append(delivery)
            return "dead_lettered"
        self._push(replace(delivery, attempt=delivery.attempt + 1))
        return "retry_scheduled"

    def recover(self, worker_id: str) -> bool:
        """Возвращает незавершённую аренду после сбоя без расхода попытки."""
        delivery = self.inflight.pop(worker_id, None)
        if delivery is None:
            return False
        # Сбой не расходует попытку: брокер повторно выдаст ту же задачу.
        self._push(delivery)
        return True


_queue_demo = PriorityTaskQueue(capacity=2)
_normal = Task("demo-normal", 5, "report:1", "обычный отчёт")
_urgent = Task("demo-urgent", 1, "incident:1", "срочный инцидент")
assert _queue_demo.publish(_normal)
assert _queue_demo.publish(_urgent)
assert not _queue_demo.publish(replace(_normal, message_id="overflow"))
_reserved_demo = _queue_demo.reserve("worker-demo")
assert _reserved_demo.task.message_id == "demo-urgent"
_queue_demo.ack("worker-demo")

pprint({
    "приоритетная_очередь": {
        "первой_выдана": _reserved_demo.task.message_id,
        "остались_готовы": [item[2].task.message_id for item in _queue_demo.ready],
        "переполнение_принято": False,
        "аренды_после_ack": _queue_demo.inflight,
    }
})

{'приоритетная_очередь': {'аренды_после_ack': {},
                          'остались_готовы': ['demo-normal'],
                          'первой_выдана': 'demo-urgent',
                          'переполнение_принято': False}}


Отправитель видит отказ при переполнении, срочная задача выдаётся раньше обычной, а задача остаётся в аренде до явного ack.

### Задание 1. Завершите один цикл обработки задачи

Реализуйте `consume_once`. Здесь только четыре результата: `idle`, `success`, `retry_scheduled` и `dead_lettered`. Не пытайтесь пока ограничивать нагрузку или реализовывать идемпотентность — они появятся отдельно.

In [4]:
def consume_once(
    queue: PriorityTaskQueue,
    worker_id: str,
    handler: Callable[[Delivery], Any],
    *,
    max_attempts: int = 3,
) -> tuple[str, int, str | None]:
    """Получает одну Delivery, обрабатывает её Task и возвращает итог попытки."""
    delivery = queue.reserve(worker_id)
    if delivery is None:
        return "idle", 0, None
    try:
        handler(delivery)
    except TransientError:
        # Ожидаемая временная ошибка расходует ограниченный запас попыток.
        status = queue.retry_or_dead_letter(worker_id, max_attempts)
        return status, delivery.attempt, delivery.task.message_id
    # Обработчик сохранил результат — аренду можно закрыть через ack.
    queue.ack(worker_id)
    return "success", delivery.attempt, delivery.task.message_id


_unavailable_queue = PriorityTaskQueue(capacity=1)
_unavailable_task = Task("m-unavailable", 2, "billing:7", "возврат")
assert _unavailable_queue.publish(_unavailable_task)


def _always_unavailable(_delivery: Delivery) -> None:
    """Имитирует временную ошибку поставщика во всех разрешённых попытках."""
    raise TransientError("поставщик вернул 503")


_unavailable_statuses = [
    consume_once(_unavailable_queue, "worker-a", _always_unavailable)[0]
    for _ in range(3)
]
assert _unavailable_statuses == ["retry_scheduled", "retry_scheduled", "dead_lettered"]
assert [item.task.message_id for item in _unavailable_queue.dead_letters] == ["m-unavailable"]

_success_queue = PriorityTaskQueue(capacity=1)
assert _success_queue.publish(replace(_unavailable_task, message_id="m-ok"))
assert consume_once(_success_queue, "worker-a", lambda _delivery: "ok")[:2] == (
    "success", 1,
)
_empty_queue = PriorityTaskQueue(capacity=1)
assert consume_once(_empty_queue, "worker-a", lambda _delivery: "unused") == (
    "idle", 0, None,
)

pprint({
    "обработка_задачи": {
        "состояния_после_временной_ошибки": _unavailable_statuses,
        "попытки_в_dlq": [_delivery.attempt for _delivery in _unavailable_queue.dead_letters],
        "успех_подтверждён": not _success_queue.inflight,
    }
})

{'обработка_задачи': {'попытки_в_dlq': [3],
                      'состояния_после_временной_ошибки': ['retry_scheduled',
                                                           'retry_scheduled',
                                                           'dead_lettered'],
                      'успех_подтверждён': True}}


Временная ошибка повторяется строго в пределах лимита и завершается записью в DLQ; успешная задача подтверждается с первой попытки.

## 2. Ограничение нагрузки (backpressure)

`max_concurrency` ограничивает число одновременных вызовов поставщика. Если место занято, задача может подождать в ограниченной очереди. Если очередь ожидания тоже заполнена, задача явно получает `dropped_overload`.

На этом занятии мы намеренно не добавляем ограничения по токенам, частоте и стоимости: это та же проверка допуска, но с дополнительными счётчиками.

In [5]:
@dataclass(slots=True)
class ProviderGate:
    """Ограничивает одновременные вызовы поставщика и очередь ожидания."""

    max_concurrency: int
    buffer_limit: int
    active: dict[str, Task] = field(default_factory=dict)
    waiting: list[tuple[int, int, Task]] = field(default_factory=list)
    dropped: list[dict[str, str]] = field(default_factory=list)
    _sequence: int = 0

    def __post_init__(self) -> None:
        """Проверяет границы параллелизма и очереди ожидания."""
        if self.max_concurrency <= 0 or self.buffer_limit < 0:
            raise ValueError("некорректные ограничения поставщика")


def complete_call(gate: ProviderGate, message_id: str) -> Task | None:
    """Освобождает место поставщика и запускает приоритетную ожидающую задачу."""
    if message_id not in gate.active:
        raise ValueError("вызов не находится среди активных")
    gate.active.pop(message_id)
    if not gate.waiting:
        return None
    # Освободившееся место получает самая приоритетная ожидающая задача.
    promoted = heappop(gate.waiting)[2]
    gate.active[promoted.message_id] = promoted
    return promoted

pprint({
    "ограничитель_поставщика": {
        "решения": ["admitted", "buffered", "dropped_overload"],
        "порядок_проверок": ["max_concurrency", "buffer_limit", "отказ"],
    }
})

{'ограничитель_поставщика': {'порядок_проверок': ['max_concurrency',
                                                  'buffer_limit',
                                                  'отказ'],
                             'решения': ['admitted',
                                         'buffered',
                                         'dropped_overload']}}


Ограничитель поставщика отделяет активные вызовы от ограниченной очереди ожидания и переводит задачу в работу после освобождения места.

### Задание 2. Запустить, подождать или отклонить

Реализуйте три последовательные ветви `admitted → buffered → dropped_overload`. При освобождении места готовая функция `complete_call` переводит в работу самую приоритетную ожидающую задачу.

In [9]:
def admit_task(task: Task, gate: ProviderGate) -> str:
    """Решает: запустить задачу, поставить в ожидание или отклонить."""
    if len(gate.active) < gate.max_concurrency:
        gate.active[task.message_id] = task
        return "admitted"
    if len(gate.waiting) < gate.buffer_limit:
        gate._sequence += 1
        # Очередь ожидания использует тот же принцип приоритета/FIFO, что брокер.
        heappush(gate.waiting, (task.priority, gate._sequence, task))
        return "buffered"
    # Явный отказ лучше молчаливой потери: причину увидят отправитель и метрики.
    gate.dropped.append({"message_id": task.message_id, "reason": "overload"})
    return "dropped_overload"


_gate = ProviderGate(max_concurrency=1, buffer_limit=1)
_call_1 = Task("call-1", 5, "call:1", "первая задача")
_call_2 = Task("call-2", 1, "call:2", "срочная задача")
_call_3 = Task("call-3", 3, "call:3", "лишняя задача")
_admission_gallery = [
    admit_task(_call_1, _gate),
    admit_task(_call_2, _gate),
    admit_task(_call_3, _gate),
]
assert _admission_gallery == ["admitted", "buffered", "dropped_overload"]
assert complete_call(_gate, "call-1").message_id == "call-2"
assert set(_gate.active) == {"call-2"}
assert _gate.dropped == [{"message_id": "call-3", "reason": "overload"}]

pprint({
    "ограничение_нагрузки": {
        "решения": _admission_gallery,
        "переведена_в_работу": "call-2",
        "активные": sorted(_gate.active),
        "отклонённые": _gate.dropped,
    }
})

{'ограничение_нагрузки': {'активные': ['call-2'],
                          'отклонённые': [{'message_id': 'call-3',
                                           'reason': 'overload'}],
                          'переведена_в_работу': 'call-2',
                          'решения': ['admitted',
                                      'buffered',
                                      'dropped_overload']}}


Свободное место запускает вызов, следующая задача ожидает, а заполненная очередь даёт наблюдаемый отказ вместо молчаливой потери.

## 3. Повторная доставка без повторного выполнения бизнес-операции

Два ключа решают разные задачи:

- `message_id` узнаёт повтор конкретного сообщения;
- `idempotency_key` узнаёт ту же бизнес-команду даже в новом сообщении.

Обработчик сохраняет бизнес-операцию и результат до возврата. Только после возврата `consume_once` делает `ack`. Поэтому сбой между сохранением и `ack` приводит к безопасной повторной доставке.

### Задание 3. Обработчик, устойчивый к сбою

Реализуйте `handle_delivery`: сначала устраните повтор по `message_id`, затем сохраните бизнес-операцию через `setdefault(idempotency_key, ...)`, затем результат. `ack` внутри обработчика не нужен — им уже управляет цикл обработки.

In [10]:
def handle_delivery(
    delivery: Delivery,
    outcomes: dict[str, dict[str, Any]],
    effects: dict[str, dict[str, Any]],
    *,
    crash_after_save: bool = False,
) -> str:
    """Сохраняет результат идемпотентно и при необходимости моделирует сбой."""
    task = delivery.task
    saved = outcomes.get(task.message_id)
    if saved is not None:
        if saved["idempotency_key"] != task.idempotency_key:
            raise ValueError("message_id связан с другой бизнес-командой")
        return "deduplicated"
    # Разные message_id одной команды не должны повторить бизнес-операцию.
    effects.setdefault(task.idempotency_key, {
        "status": "applied", "owner_message_id": task.message_id,
    })
    outcomes[task.message_id] = {
        "status": "success", "idempotency_key": task.idempotency_key,
    }
    # Цикл ещё не сделал ack: здесь моделируется повторная выдача той же задачи.
    if crash_after_save:
        raise WorkerCrash("сбой после сохранения, но до ack брокера")
    return "processed"


_crash_task = Task("m-crash", 1, "refund:P-77", "вернуть 1490 ₽ по P-77")
_crash_queue = PriorityTaskQueue(capacity=1)
assert _crash_queue.publish(_crash_task)
recovery_outcomes: dict[str, dict[str, Any]] = {}
recovery_effects: dict[str, dict[str, Any]] = {}

try:
    consume_once(
        _crash_queue,
        "worker-old",
        lambda delivery: handle_delivery(
            delivery, recovery_outcomes, recovery_effects, crash_after_save=True,
        ),
    )
    raise AssertionError("рабочий процесс должен остановиться до ack")
except WorkerCrash:
    pass

assert "worker-old" in _crash_queue.inflight
assert _crash_queue.recover("worker-old")
_replay = consume_once(
    _crash_queue,
    "worker-new",
    lambda delivery: handle_delivery(
        delivery, recovery_outcomes, recovery_effects,
    ),
)
assert _replay[0] == "success" and not _crash_queue.inflight

# Повтор клиента может иметь новый message_id, но ту же бизнес-команду.
_same_effect_task = replace(_crash_task, message_id="m-client-retry")
assert handle_delivery(
    Delivery(_same_effect_task), recovery_outcomes, recovery_effects,
) == "processed"
assert len(recovery_outcomes) == 2 and len(recovery_effects) == 1
assert recovery_outcomes["m-crash"] == {
    "status": "success", "idempotency_key": "refund:P-77",
}
assert recovery_effects["refund:P-77"] == {
    "status": "applied", "owner_message_id": "m-crash",
}

pprint({
    "восстановление_после_сбоя": {
        "повторная_выдача_задачи": _replay,
        "результаты": recovery_outcomes,
        "бизнес_операции": recovery_effects,
        "число_операций": len(recovery_effects),
        "аренды_после_ack": _crash_queue.inflight,
    }
})

{'восстановление_после_сбоя': {'аренды_после_ack': {},
                               'бизнес_операции': {'refund:P-77': {'owner_message_id': 'm-crash',
                                                                   'status': 'applied'}},
                               'повторная_выдача_задачи': ('success',
                                                           1,
                                                           'm-crash'),
                               'результаты': {'m-client-retry': {'idempotency_key': 'refund:P-77',
                                                                 'status': 'success'},
                                              'm-crash': {'idempotency_key': 'refund:P-77',
                                                          'status': 'success'}},
                               'число_операций': 1}}


Повторно выданная задача находит сохранённый результат и не повторяет операцию; новый message_id той же команды защищён отдельным idempotency_key.

## 4. Собираем один проход рабочего процесса

Осталось связать готовые функции. Каркас уже обрабатывает `idle`, проверяет нагрузку до `reserve`, создаёт вложенный handler, гарантированно освобождает provider slot через `finally` и собирает итоговое событие. Заполните только вызов идемпотентного `handle_delivery` внутри handler.

Порядок важен: если поставщик занят, не вызывайте `reserve`. Тогда задача остаётся только в очереди брокера и не имеет двух владельцев одновременно. В этом режиме именно очередь брокера служит местом ожидания, поэтому `ProviderGate.waiting` не используется вторично.

In [11]:
def run_worker_once(
    queue: PriorityTaskQueue,
    worker_id: str,
    gate: ProviderGate,
    outcomes: dict[str, dict[str, Any]],
    effects: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    """Выполняет один проход worker с проверкой нагрузки до reserve."""
    if not queue.ready:
        return {"message_id": None, "status": "idle", "attempt": 0}
    if len(gate.active) >= gate.max_concurrency:
        # Не делаем reserve: задача остаётся только в очереди брокера.
        waiting_delivery = queue.ready[0][2]
        return {
            "message_id": waiting_delivery.task.message_id,
            "status": "backpressured",
            "attempt": waiting_delivery.attempt,
        }
    event: dict[str, Any] = {}

    def handler(delivery: Delivery) -> None:
        """Передаёт Task из Delivery идемпотентному обработчику после допуска."""
        task = delivery.task
        if admit_task(task, gate) != "admitted":
            raise AssertionError("свободный рабочий процесс должен получить место")
        try:
            event["handler_status"] = handle_delivery(
                delivery, outcomes, effects,
            )
        finally:
            # Место поставщика освобождается даже при ошибке внутри обработчика.
            complete_call(gate, task.message_id)

    status, attempt, message_id = consume_once(queue, worker_id, handler)
    event.update({"message_id": message_id, "status": status, "attempt": attempt})
    return event


_runtime_queue = PriorityTaskQueue(capacity=2)
_runtime_normal = Task("batch-normal", 5, "report:normal", "обычная задача")
_runtime_urgent = Task("batch-urgent", 1, "report:urgent", "срочная задача")
assert _runtime_queue.publish(_runtime_normal)
assert _runtime_queue.publish(_runtime_urgent)

integrated_outcomes: dict[str, dict[str, Any]] = {}
integrated_effects: dict[str, dict[str, Any]] = {}
_runtime_gate = ProviderGate(max_concurrency=1, buffer_limit=1)
_busy_call = Task("provider-busy", 0, "system:busy", "поставщик занят")
assert admit_task(_busy_call, _runtime_gate) == "admitted"

_worker_events = [run_worker_once(
    _runtime_queue, "worker-a", _runtime_gate,
    integrated_outcomes, integrated_effects,
)]
assert _worker_events[-1]["status"] == "backpressured"
assert not _runtime_queue.inflight and len(_runtime_queue.ready) == 2
complete_call(_runtime_gate, "provider-busy")

while _runtime_queue.ready:
    _worker_events.append(run_worker_once(
        _runtime_queue, "worker-a", _runtime_gate,
        integrated_outcomes, integrated_effects,
    ))

_processed_order = [
    event["message_id"] for event in _worker_events
    if event["status"] == "success"
]
assert _processed_order == ["batch-urgent", "batch-normal"]
assert not _runtime_gate.active and not _runtime_queue.inflight
assert len(integrated_outcomes) == len(integrated_effects) == 2

task_catalog = {
    task.message_id: task
    for task in (_crash_task, _same_effect_task, _runtime_normal, _runtime_urgent)
}

pprint({
    "рабочий_процесс": {
        "события": _worker_events,
        "порядок_обработки": _processed_order,
        "результаты": sorted(integrated_outcomes),
        "бизнес_операции": sorted(integrated_effects),
        "активные_вызовы_после_прохода": sorted(_runtime_gate.active),
        "аренды_после_прохода": _runtime_queue.inflight,
    }
})

{'рабочий_процесс': {'активные_вызовы_после_прохода': [],
                     'аренды_после_прохода': {},
                     'бизнес_операции': ['report:normal', 'report:urgent'],
                     'порядок_обработки': ['batch-urgent', 'batch-normal'],
                     'результаты': ['batch-normal', 'batch-urgent'],
                     'события': [{'attempt': 1,
                                  'message_id': 'batch-urgent',
                                  'status': 'backpressured'},
                                 {'attempt': 1,
                                  'handler_status': 'processed',
                                  'message_id': 'batch-urgent',
                                  'status': 'success'},
                                 {'attempt': 1,
                                  'handler_status': 'processed',
                                  'message_id': 'batch-normal',
                                  'status': 'success'}]}}


Рабочий процесс оставляет задачу в очереди при занятом поставщике, а после допуска выполняет порядок reserve → сохранить → ack.

## GigaChat: почему повторная задача безопасна

Запускайте эту ячейку после выполнения всех четырёх заданий и настройки `.env` или Colab Secrets. Она использует один инструмент только для чтения. GigaChat сначала запрашивает сохранённый результат конкретной задачи, приложение возвращает наблюдение как `ToolMessage`, и только затем модель объясняет, почему повтор безопасен. Решения об `ack`, новой попытке и отказе остаются в коде Python.

In [12]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_gigachat.chat_models import GigaChat

load_dotenv(Path("../.env").resolve())


@tool
def inspect_delivery(message_id: str) -> dict:
    """Получить сохранённый результат задачи и данные попытки её выдачи."""
    # Инструмент только читает данные и не может делать ack, повтор или отказ.
    selected = task_catalog.get(message_id)
    if selected is None:
        return {"message_id": message_id, "status": "not_found"}
    outcomes = {**recovery_outcomes, **integrated_outcomes}
    effects = {**recovery_effects, **integrated_effects}
    saved = outcomes.get(message_id)
    effect = effects.get(selected.idempotency_key)
    return {
        "message_id": message_id,
        "состояние_задачи": saved["status"] if saved else "not_completed",
        "ключ_идемпотентности": selected.idempotency_key,
        "бизнес_операция": effect,
        "приоритет": selected.priority,
        "содержимое": selected.payload,
    }


def build_lab4_gigachat(*, tool_choice: str):
    """Создаёт GigaChat с единственным read-only инструментом проверки задачи."""
    # Одна функция гарантирует одинаковые параметры для обоих обращений к модели.
    credentials = os.getenv("GIGACHAT_CREDENTIALS")
    if not credentials:
        raise RuntimeError(
            "задайте GIGACHAT_CREDENTIALS в корневом .env или в Colab Secrets"
        )
    kwargs = {
        "credentials": credentials,
        "scope": os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_B2B"),
        "model": os.getenv("GIGACHAT_MODEL", "GigaChat-2-Max"),
        "verify_ssl_certs": False
    }
    if os.getenv("GIGACHAT_CA_BUNDLE_FILE"):
        kwargs["ca_bundle_file"] = os.environ["GIGACHAT_CA_BUNDLE_FILE"]
    if os.getenv("GIGACHAT_BASE_URL"):
        kwargs["base_url"] = os.environ["GIGACHAT_BASE_URL"]
    return GigaChat(**kwargs).bind_tools(
        [inspect_delivery], tool_choice=tool_choice,
    )


_delivery_question = HumanMessage(content=(
    "Проверь задачу m-crash через inspect_delivery. После получения данных "
    "объясни, почему повтор не создал вторую бизнес-операцию. "
    "Не выполняй ack и повтор самостоятельно."
))
# При первом обращении инструмент выбран принудительно: модель должна создать
# типизированный Function Call, а не придумывать объяснение без данных.
_tool_response = build_lab4_gigachat(tool_choice="inspect_delivery").invoke(
    [_delivery_question]
)
if len(_tool_response.tool_calls) != 1:
    raise RuntimeError("GigaChat-2-Max не вернул единственный вызов inspect_delivery")
_delivery_call = _tool_response.tool_calls[0]
# Инструмент исполняет приложение; модель не получает прямой доступ к Python.
_delivery_observation = inspect_delivery.invoke(_delivery_call["args"])
# Второе обращение возвращает наблюдение с тем же tool_call_id и только после
# этого просит модель сформулировать итоговое объяснение.
_delivery_diagnosis = build_lab4_gigachat(tool_choice="auto").invoke([
    _delivery_question,
    _tool_response,
    ToolMessage(
        content=json.dumps(_delivery_observation, ensure_ascii=False, default=list),
        tool_call_id=_delivery_call["id"],
    ),
])
# Новый вызов инструмента здесь означает незавершённый цикл, поэтому это ошибка.
if _delivery_diagnosis.tool_calls:
    raise RuntimeError("после наблюдения ожидался итоговый ответ без нового вызова")

pprint({
    "сетевой_вывод_gigachat": {
        "источник": "gigachat_online",
        "вызов_инструмента": _delivery_call,
        "наблюдение": _delivery_observation,
        "объяснение": _delivery_diagnosis.content,
        "неожиданные_повторные_вызовы": _delivery_diagnosis.tool_calls,
    }
})

{'сетевой_вывод_gigachat': {'вызов_инструмента': {'args': {'message_id': 'm-crash'},
                                                  'id': 'a0129173-b413-4ab6-8409-7c7ef2602027',
                                                  'name': 'inspect_delivery',
                                                  'type': 'tool_call'},
                            'источник': 'gigachat_online',
                            'наблюдение': {'message_id': 'm-crash',
                                           'бизнес_операция': {'owner_message_id': 'm-crash',
                                                               'status': 'applied'},
                                           'ключ_идемпотентности': 'refund:P-77',
                                           'приоритет': 1,
                                           'содержимое': 'вернуть 1490 ₽ по '
                                                         'P-77',
                                           'состояние_задачи': 'success'},
    

GigaChat объясняет повтор по данным только для чтения, но не получает полномочий менять состояние брокера.

## После занятия: как расширять среду выполнения

Эти темы полезны, но не нужны для сегодняшних четырёх заданий:

- **шардирование** — несколько физических очередей и один владелец каждой;
- **пошаговый оркестратор** — план, ограничение времени шага и сохранённая позиция;
- **квоты поставщика** — ограничения запросов, токенов и стоимости перед внешним вызовом;
- **контрольные точки LangGraph** — остановка и возобновление через устойчивое хранилище.

Добавляйте их по одной только после того, как базовый порядок `reserve → сохранить → ack` стал понятен.

## Вся обязательная схема

```text
отправитель → Task → очередь → reserve → Delivery → рабочий процесс
                                                   |
                               сохранить результат → ack → завершено

временная ошибка → повтор → очередь
попытки исчерпаны → DLQ
сбой до ack → recover → очередь
```

У повторной попытки и повторной выдачи задачи разные причины. Новая попытка отвечает на временную ошибку обработки; повторная выдача после сбоя проверяет уже сохранённый результат.

## Итоги

### Что было сделано

Вы довели сквозной путь до надёжного выполнения: агент предложил действие, приложение ограничило его правилами, а рабочий процесс получил подтверждённую задачу через очередь. Приоритет, повторы и DLQ управляют попытками выполнения; ограничение нагрузки защищает поставщика; `idempotency_key` не даёт повторить бизнес-операцию после сбоя.

Четыре ноутбука теперь образуют одну цепочку: `tool call → ограниченный цикл → навык и дочерний агент → надёжное выполнение подтверждённой задачи`.

### Чему вы научились

- отличать `reserve` от окончательного `ack`;
- ограничивать число одновременных вызовов поставщика и размер очереди ожидания;
- различать устранение дублей сообщения и идемпотентность бизнес-операции;
- оставлять задачу в очереди брокера при занятом поставщике.

### Контрольные вопросы

1. Почему `ack` нельзя делать до сохранения результата?
2. Чем `buffered` отличается от молчаливой потери сообщения?
3. Зачем нужны и `message_id`, и `idempotency_key`?
4. Почему рабочий процесс проверяет доступность поставщика до `reserve`?

# Приложение — короткая памятка

Открывайте после самостоятельного решения.

<details>
<summary>Показать четыре инварианта</summary>

1. Аренда живёт у брокера до `ack`.
2. Только ожидаемая временная ошибка расходует запас попыток.
3. Перегрузка заканчивается наблюдаемым статусом, а не потерей задачи.
4. `idempotency_key` защищает бизнес-операцию между разными сообщениями одной команды.

В рабочей системе учебные структуры будут заменены настоящим брокером и устойчивым хранилищем, но порядок операций останется тем же.

</details>